<a href="https://colab.research.google.com/github/Kimballa05/agentic-ai-jetstream2-bioinformatics/blob/main/agentic_ai_bioinformatics_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Environment Setup** (Ideally these should already be set up in Jetstream2 image)

In [1]:
# Install a small tool set for the Colab prototype
!apt-get update -qq
!apt-get install -y -qq sra-toolkit fastqc fastp pigz > /dev/null
!pip install -q multiqc pandas scikit-learn pyyaml

print("Environment setup complete.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 951.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [2]:
# Check tool versions
!fasterq-dump --version | head -n 2
!fastqc --version
!fastp --version 2>&1 | head -n 1
!multiqc --version


fasterq-dump : 3.0.3 ( 3.0.2 )
[0.005s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.005s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
FastQC v0.12.1
fastp 0.23.4
multiqc, version 1.35


**Create a workspace**

The original reads go in raw/ and should never be overwritten.

In [3]:
from pathlib import Path

base = Path("/content/agentic_bioinfo")

for directory in [
    "raw",
    "clean",
    "qc/pre",
    "qc/post",
    "results",
    "logs",
    "reports",
]:
    (base / directory).mkdir(parents=True, exist_ok=True)

print(f"Workspace created at: {base}")

Workspace created at: /content/agentic_bioinfo


**Retrieve the raw sequencing data**

In the final agentic workflow, the participant provides the accession and the agent performs the retrieval.
User: "Retrieve ERR4567643, convert to paired FASTQ files, compress, then put in the 'raw' directory"

Explicit execution of the tool calls

In [4]:
ACCESSION = "ERR4567643"

# Download into the SRA cache
!prefetch {ACCESSION}

# Convert to paired FASTQ files
!fasterq-dump {ACCESSION} \
    --split-files \
    --outdir /content/agentic_bioinfo/raw

# Compress FASTQ files
!pigz -f /content/agentic_bioinfo/raw/*.fastq

print("Retrieval step finished.")


2026-10-05T15:41:35 prefetch.3.0.3: Current preference is set to retrieve SRA Normalized Format files with full base quality scores.
2026-10-05T15:41:35 prefetch.3.0.3: 1) Downloading 'ERR4567643'...
2026-10-05T15:41:35 prefetch.3.0.3: SRA Normalized Format file is being retrieved, if this is different from your preference, it may be due to current file availability.
2026-10-05T15:41:35 prefetch.3.0.3:  Downloading via HTTPS...
2026-10-05T15:41:38 prefetch.3.0.3:  HTTPS download succeed
2026-10-05T15:41:38 prefetch.3.0.3:  'ERR4567643' is valid
2026-10-05T15:41:38 prefetch.3.0.3: 1) 'ERR4567643' was downloaded successfully
2026-10-05T15:41:38 prefetch.3.0.3: 'ERR4567643' has 0 unresolved dependencies
spots read      : 410,740
reads read      : 821,480
reads written   : 821,480
Retrieval step finished.
